# Text Summarizer

Fine-tune T5-small on the SamSum dialogue summarization dataset.

In [1]:
from pathlib import Path
import re
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
ARTIFACT_DIR = Path("artifacts")
MODEL_DIR = ARTIFACT_DIR / "saved_summary_model"

MODEL_NAME = "t5-small"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 150
TRAIN_SAMPLES = 4000
VALIDATION_SAMPLES = 500


## Load the correct dataset splits

In [2]:
train_data = pd.read_csv(DATA_DIR / "samsum-train.csv")
val_data = pd.read_csv(DATA_DIR / "samsum-validation.csv")
test_data = pd.read_csv(DATA_DIR / "samsum-test.csv")

train_data = train_data.sample(
    n=min(TRAIN_SAMPLES, len(train_data)),
    random_state=42,
).reset_index(drop=True)

val_data = val_data.sample(
    n=min(VALIDATION_SAMPLES, len(val_data)),
    random_state=42,
).reset_index(drop=True)

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)


Train: (4000, 3)
Validation: (500, 3)
Test: (819, 3)


## Clean the text

In [3]:
def clean_data(text: str) -> str:
    text = re.sub(r"\r\n?", " ", str(text))
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    return text.strip().lower()

for frame in (train_data, val_data, test_data):
    frame["dialogue"] = frame["dialogue"].map(clean_data)
    frame["summary"] = frame["summary"].map(clean_data)


## Tokenize for T5 summarization

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_batch(batch):
    inputs = [f"summarize: {text}" for text in batch["dialogue"]]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = Dataset.from_pandas(train_data[["dialogue", "summary"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_data[["dialogue", "summary"]], preserve_index=False)

train_dataset = train_dataset.map(preprocess_batch, batched=True, remove_columns=["dialogue", "summary"])
val_dataset = val_dataset.map(preprocess_batch, batched=True, remove_columns=["dialogue", "summary"])


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## Load the T5 model

In [5]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Seq2SeqTrainer handles device placement, so the model does not need
# to be moved manually before training.


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Train

In [ ]:
from inspect import signature

args = dict(
    output_dir=str(ARTIFACT_DIR / "trainer_output"),
    num_train_epochs=6,
    learning_rate=5e-5,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    report_to="none",
)

parameters = signature(Seq2SeqTrainingArguments).parameters

if "eval_strategy" in parameters:
    args["eval_strategy"] = "epoch"
else:
    args["evaluation_strategy"] = "epoch"

training_args = Seq2SeqTrainingArguments(**args)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.827450,1.636862


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Save the trained model

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f"Saved model to: {MODEL_DIR.resolve()}")


## Test inference

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu"
)

model.to(device)
model.eval()

def summarize_dialogue(dialogue: str) -> str:
    cleaned = clean_data(dialogue)
    inputs = tokenizer(
        f"summarize: {cleaned}",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        return_tensors="pt",
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.inference_mode():
        targets = model.generate(
            **inputs,
            num_beams=4,
            max_length=MAX_TARGET_LENGTH,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    return tokenizer.decode(targets[0], skip_special_tokens=True).strip()

t_inp = """Who are you voting for in this election?\r\nOliver: Liberals as always.\r\nOlivia: Me too!!\r\n"""
print(summarize_dialogue(t_inp))
